# 2. MusSkl donor clustering

Part of the **[Fig. 6 chapter](fig6.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import cooler
import anndata
import scanpy as sc
import scanpy.external as sce
from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# mpl.use('agg')
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
outdir = f'{ENTEX_ROOT}/analysis/pairwise_prediction/'


In [ ]:
ct = 'Mus-Skl'


In [ ]:
npc = pd.read_csv(f'{indir}npc.tsv', sep='\t', header=None, index_col=0, names=['npc_cg', 'npc_3c'])

In [ ]:
adata_mc = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG_embed.h5ad')
adata_3c = anndata.read_h5ad(f'{indir}L2/{ct}/100k3C_embed.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [ ]:
adata = anndata.read_h5ad('Mus-Skl/5kCG100k3C_embed.h5ad')
adata

In [ ]:
adata.obs['Donor'].value_counts()

## Using uncorrected embedding


In [ ]:
def rename_cluster(xx):
    yy = xx.replace('-','/').replace('mc','').replace('3c','')
    yy = yy.replace('0', 'Stem').replace('1', 'Fast').replace('2', 'Slow')
    return yy

leg = np.sort(adata.obs['group_mc_3c'].unique())
k1, k2 = 0, 0
color_palette = {}
for xx in leg:
    x1, x2 = xx.split('-')
    if x1[1:]==x2[1:]:
        color_palette[rename_cluster(xx)] = sns.color_palette('tab20')[k1*2+1]
        k1 += 1
    else:
        color_palette[rename_cluster(xx)] = sns.color_palette('tab20')[k2*2]
        k2 += 1


In [ ]:
adata_list = []
for obsm_key, raw_key, adata_tmp in zip(['X_lsi', 'X_pca'], ['5kCG_lsi', '100k3C_pca'], [adata_mc, adata_3c]):
    for xx in adata.obs['Donor'].unique():
        tmp = adata_tmp[adata_tmp.obs['Donor']==xx].copy()
        # npc = significant_pc_test(tmp, p_cutoff=0.05, obsm=obsm_key, update=False)
        # tmp.obsm[obsm_key] = normalize(tmp.obsm[raw_key][:, :10], axis=1)
        tsne(tmp, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
        sc.pp.neighbors(tmp, n_neighbors=25, use_rep=obsm_key)
        sc.tl.leiden(tmp, resolution=0.5, random_state=0, flavor='igraph')
        adata_list.append(tmp.copy())
        print(obsm_key, xx)
        

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list):
    tmp.obs['group_mc_3c'] = adata.obs['group_mc_3c'].astype(str)
    tmp.obs['group_mc_3c'] = tmp.obs['group_mc_3c'].map(rename_cluster)
    dump_embedding(tmp, coord_base)
    ax = axes.flatten()[i]
    idx_row, idx_col = i//2, i%2
    ax.set_title(f'{["mC","3C"][idx_row]} {adata.obs["Donor"].unique()[idx_col]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='group_mc_3c',
                            # text_anno='group_mc_3c', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette=color_palette,
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            show_legend=True
                            )


In [ ]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 0.1
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 50

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 32

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [ ]:
for tmp in adata_list:
    sc.tl.leiden(tmp, resolution=0.1, random_state=0, flavor='igraph')


In [ ]:
for i,obsm_key in enumerate(['X_lsi', 'X_pca']):
    for j,xx in enumerate(adata.obs['Donor'].unique()):
        tmp = adata_list[i*2+j]
        cc.fit_predict(tmp.obsm[obsm_key])
        tmp.obs['consensus_leiden_init'] = cc.label.copy()


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list):
    dump_embedding(tmp, coord_base)
    ax = axes.flatten()[i]
    idx_row, idx_col = i//2, i%2
    ax.set_title(f'{["mC","3C"][idx_row]} {adata.obs["Donor"].unique()[idx_col]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='consensus_leiden_init',
                            text_anno='consensus_leiden_init', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                            )


In [ ]:
for i in range(2):
    for j in range(2):
        tmp = adata_list[i*2+j]
        tmp.obs['consensus_leiden_other'] = adata_list[(1-i)*2+j].obs['consensus_leiden_init'].values
        

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list):
    dump_embedding(tmp, coord_base)
    ax = axes.flatten()[i]
    idx_row, idx_col = i//2, i%2
    ax.set_title(f'{["mC","3C"][idx_row]} {adata.obs["Donor"].unique()[idx_col]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='consensus_leiden_other',
                            text_anno='consensus_leiden_other', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                            )


In [ ]:
adata_list[0].obs[['consensus_leiden_init','consensus_leiden_other']].value_counts().unstack()

In [ ]:
adata_list[1].obs[['consensus_leiden_init','consensus_leiden_other']].value_counts().unstack().fillna(0).astype(int)

## Using donor separated embedding

In [ ]:
var_dim = 'chrom5k'
chrom_to_remove = ['chrX', 'chrY', 'chrM', 'chrL']
mcds_path_list = glob(f'{ENTEX_ROOT}/mcds/*mcds')
mcds = MCDS.open(mcds_path_list, var_dim=var_dim)
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel(cell=adata.obs.index)
mcds

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo')
binarize_matrix(mcad, cutoff=0.95)


In [ ]:
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=black_list_path)


In [ ]:
mcad.write_h5ad(f'{indir}L2/{ct}/5kCG.h5ad')

In [ ]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [ ]:
raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
adata_mc_list = []
for xx in adata.obs['Donor'].unique():
    tmp = mcad[adata.obs['Donor']==xx].copy()
    model.fit_transform(tmp, obsm_name='5kCG_lsi')
    npc = significant_pc_test(tmp, p_cutoff=0.1, obsm=raw_key, update=False)
    tmp.obsm[obsm_key] = normalize(tmp.obsm[raw_key][:, :npc], axis=1)
    tsne(tmp, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
    adata_mc_list.append(tmp.copy())
    print(obsm_key, xx)
    

In [ ]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
indir = f'{ENTEX_ROOT}/'

In [ ]:

dim = 50
cell_list = pd.read_csv(f'{indir}embedding/SkGcn/cell_table.tsv', index_col=0, header=None, sep='\t')
cell_list['index'] = np.arange(cell_list.shape[0])
matrix = [[], []]
for chrom in chrom_sizes.index:
    data = np.load(f'{indir}embedding/SkGcn/{chrom}.npz')['arr_0']
    for i,xx in enumerate(adata.obs['Donor'].unique()):
        selc = adata.obs.index[adata.obs['Donor']==xx]
        tmp = data[cell_list.loc[selc, 'index'].values]
        dim = min(dim, tmp.shape[0] - 1, tmp.shape[1] - 1)
        model = TruncatedSVD(n_components=dim, algorithm='arpack')
        tmp_reduce = model.fit_transform(tmp)
        matrix[i].append(tmp_reduce / model.singular_values_)
    print(chrom)


In [ ]:
raw_key = '100k3C_pca'
obsm_key = 'X_pca'
model = TruncatedSVD(n_components=dim, algorithm='arpack')
adata_3c_list = []
for i,xx in enumerate(adata.obs['Donor'].unique()):
    selc = adata.obs.index[adata.obs['Donor']==xx]
    tmp = anndata.AnnData(X=np.concatenate(matrix[i], axis=1), obs=adata.obs.loc[selc].copy())
    tmp.obsm[raw_key] = model.fit_transform(tmp.X)
    tmp.uns['100k3C_eigen'] = model.singular_values_.copy()
    npc = significant_pc_test(tmp, p_cutoff=0.1, obsm=raw_key, update=False)
    tmp.obsm[obsm_key] = normalize(tmp.obsm[raw_key][:, :npc], axis=1)
    tsne(tmp, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
    adata_3c_list.append(tmp.copy())
    print(obsm_key, xx)


In [ ]:
for i,xx in enumerate(adata.obs['Donor'].unique()):
    tmp = adata_3c_list[i].copy()
    tmp = anndata.AnnData(obs=tmp.obs, obsm=tmp.obsm)
    tmp.write_h5ad(f'Mus-Skl/100k3C_{xx}_embed.h5ad')
    tmp = adata_mc_list[i].copy()
    tmp = anndata.AnnData(obs=tmp.obs, obsm=tmp.obsm)
    tmp.write_h5ad(f'Mus-Skl/5kCG_{xx}_embed.h5ad')


In [ ]:
adata_list = []

for mode in ['5kCG', '100k3C']:
    for xx in adata.obs['Donor'].unique():
        tmp = anndata.read_h5ad(f'Mus-Skl/{mode}_{xx}_embed.h5ad')
        adata_list.append(tmp.copy())
        print(mode, xx)
        

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list[:2]):
    tmp.obs['group_mc_3c'] = adata.obs['group_mc_3c'].astype(str)
    tmp.obs['group_mc_3c'] = tmp.obs['group_mc_3c'].map(rename_cluster)
    dump_embedding(tmp, coord_base)
    ax = axes[0,i]
    ax.set_title(f'mC {adata.obs["Donor"].unique()[i]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='group_mc_3c',
                            # text_anno='group_mc_3c', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette=color_palette,
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            show_legend=True
                            )

for i, tmp in enumerate(adata_list[2:]):
    tmp.obs['group_mc_3c'] = adata.obs['group_mc_3c'].astype(str)
    tmp.obs['group_mc_3c'] = tmp.obs['group_mc_3c'].map(rename_cluster)
    dump_embedding(tmp, coord_base)
    ax = axes[1,i]
    ax.set_title(f'3C {adata.obs["Donor"].unique()[i]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='group_mc_3c',
                            # text_anno='group_mc_3c', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette=color_palette,
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            show_legend=True
                            )


In [ ]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 0.1
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 50

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 32

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [ ]:
# for i,obsm_key in enumerate(['X_lsi', 'X_pca']]):
i = 1
obsm_key = 'X_pca'
for j,xx in enumerate(adata.obs['Donor'].unique()):
    tmp = adata_list[i*2+j]
    cc.fit_predict(tmp.obsm[obsm_key])
    tmp.obs['consensus_leiden_init'] = cc.label.copy()


In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list):
    dump_embedding(tmp, coord_base)
    ax = axes.flatten()[i]
    idx_row, idx_col = i//2, i%2
    ax.set_title(f'{["mC","3C"][idx_row]} {adata.obs["Donor"].unique()[idx_col]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='consensus_leiden_init',
                            text_anno='consensus_leiden_init', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                            )


In [ ]:
for i in range(2):
    for j in range(2):
        tmp = adata_list[i*2+j]
        tmp.obs['consensus_leiden_other'] = adata_list[(1-i)*2+j].obs['consensus_leiden_init'].values
        

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

for i, tmp in enumerate(adata_list):
    dump_embedding(tmp, coord_base)
    ax = axes.flatten()[i]
    idx_row, idx_col = i//2, i%2
    ax.set_title(f'{["mC","3C"][idx_row]} {adata.obs["Donor"].unique()[idx_col]}')
    _ = categorical_scatter(data=tmp.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='consensus_leiden_other',
                            text_anno='consensus_leiden_other', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                            )


In [ ]:
adata_list[0].obs[['consensus_leiden_init','consensus_leiden_other']].value_counts().unstack().fillna(0).astype(int)

In [ ]:
adata_list[1].obs[['consensus_leiden_init','consensus_leiden_other']].value_counts().unstack().fillna(0).astype(int)

In [ ]:
adata.obs['leiden_group_mc'] = 'm' + pd.concat([adata_list[0].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c2','c1':'c1'}), adata_list[1].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c1','c1':'c2'})])
adata.obs['leiden_group_3c'] = '3' + pd.concat([adata_list[2].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c2','c1':'c1'}), adata_list[3].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c1','c1':'c2'})])


In [ ]:
count = adata.obs[['leiden_group_mc', 'leiden_group_3c']].value_counts().unstack()
count

In [ ]:
adata.obs['group_mc_3c_old'] = adata.obs['group_mc_3c'].copy()
adata.obs['group_mc_3c_old'] = adata.obs['group_mc_3c_old'].map(rename_cluster)

In [ ]:
adata.obs['group_mc_3c'] = adata.obs['leiden_group_mc'].astype(str) + '-' + adata.obs['leiden_group_3c'].astype(str)
adata.obs['group_mc_3c'] = adata.obs['group_mc_3c'].map(rename_cluster)

In [ ]:
data = adata.obs.loc[~adata.obs['group_mc_3c'].isin(['Fast/Stem', 'Slow/Stem'])]
data['group_mc_3c'] = data['group_mc_3c'].astype(str)


In [ ]:
adata

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4, 2.5), dpi=300, sharex='all')
for i, xx in enumerate(['FinalmCReads', 'CisLongContact']):
    ax = axes[i]
    sns.violinplot(data, x='group_mc_3c', y=xx, ax=ax,
                   inner='box',
                   inner_kws=dict(box_width=4))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.yaxis.set_major_formatter(plt.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

fig.tight_layout()
fig.savefig('Mus-Skl/plot/mc3cgroup_totalreads_violin.pdf', transparent=True)

In [ ]:
ds = 4
coord_base = 'tsne'
fig, axes = plt.subplots(2, 2, figsize=(8,5), dpi=300)
for i,adata_tmp in enumerate([adata_mc, adata_3c]):
    tmp = adata_tmp.obs.copy()
    tmp['group_mc_3c'] = adata.obs['group_mc_3c'].astype(str)
    tmp['group_mc_3c'] = tmp['group_mc_3c'].map(rename_cluster)
    ax = axes[i,0]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='group_mc_3c',
                            # text_anno='group_mc_3c', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette=color_palette,
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            show_legend=True
                            )
    ax = axes[i,1]
    _ = continuous_scatter(data=tmp,
                           ax=ax,
                           coord_base=coord_base,
                           hue=adata.obs['mCGFrac'],
                           s=ds,
                           cmap='Blues_r',
                           labelsize=8,
                           max_points=None,
                           scatter_kws={'rasterized':True},
                           hue_norm=[0.69, 0.78],
                           )

fig.tight_layout()
fig.savefig('Mus-Skl/mC_3C_embed_group_mCG_new.pdf', transparent=True)


In [ ]:
for i,mode in enumerate(['5kCG', '100k3C']):
    for j,xx in enumerate(adata.obs['Donor'].unique()):
        tmp = adata_list[i*2+j].copy()
        tmp.write_h5ad(f'Mus-Skl/{mode}_{xx}_embed.h5ad')

In [ ]:
adata_list = []
for i,mode in enumerate(['5kCG', '100k3C']):
    for j,xx in enumerate(adata.obs['Donor'].unique()):
        tmp = anndata.read_h5ad(f'Mus-Skl/{mode}_{xx}_embed.h5ad')
        adata_list.append(tmp)


In [ ]:
adata.obs['leiden_group_mc'] = 'm' + pd.concat([adata_list[0].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c2','c1':'c1'}), adata_list[1].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c1','c1':'c2'})])
adata.obs['leiden_group_3c'] = '3' + pd.concat([adata_list[2].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c2','c1':'c1'}), adata_list[3].obs['consensus_leiden_init'].astype(str).map({'c2':'c0','c0':'c1','c1':'c2'})])


In [ ]:
adata.obs['group_mc_3c_old'] = adata.obs['group_mc_3c'].copy()
adata.obs['group_mc_3c_old'] = adata.obs['group_mc_3c_old'].map(rename_cluster)

In [ ]:
adata.obs['group_mc_3c'] = adata.obs['leiden_group_mc'].astype(str) + '-' + adata.obs['leiden_group_3c'].astype(str)
adata.obs['group_mc_3c'] = adata.obs['group_mc_3c'].map(rename_cluster)

In [ ]:
pd.DataFrame([adata.obs['group_mc_3c_old'].str.split('/').str[0], adata.obs['group_mc_3c'].str.split('/').str[0]], index=['old', 'new']).T.value_counts()

In [ ]:
ct1, ct2 = 'Fast', 'Stem'
# ct1, ct2 = 'Slow', 'Fast'
selc = adata.obs.index[(adata.obs['group_mc_3c_old'].str.split('/').str[0]==ct1) & (adata.obs['group_mc_3c'].str.split('/').str[0]==ct2)]
np.savetxt(f'Mus-Skl/allc/allclist_mc{ct1}_mc{ct2}.txt', f'{ENTEX_ROOT}/allc/' + selc + '.CGN-Both.allc.tsv.gz', fmt='%s')


In [ ]:
print((1957+1234+496)/3973, (125+40)/(125+90+40+24+6+1))

In [ ]:
pd.DataFrame([adata.obs['group_mc_3c_old'].str.split('/').str[1], adata.obs['group_mc_3c'].str.split('/').str[1]], index=['old', 'new']).T.value_counts()

In [ ]:
print((2043+1683+136)/3973, (54+50)/(54+50+7))

In [ ]:
(adata.obs['group_mc_3c_old']==adata.obs['group_mc_3c']).sum()/3973

In [ ]:
adata.write_h5ad('Mus-Skl/5kCG100k3C_embed_new.h5ad')

In [ ]:
adata = anndata.read_h5ad('Mus-Skl/5kCG100k3C_embed_new.h5ad')
adata

In [ ]:
adata.obs[['group_mc_3c', 'group_mc_3c_old']]

In [ ]:
adata.obs[['group_mc_3c_old', 'group_mc_3c']].value_counts().sort_index()

In [ ]:
adata.obs[['group_mc_3c', 'group_mc_3c_old']].value_counts().sort_index()

In [ ]:
allc_table = pd.read_csv(f'{ENTEX_ROOT}/allclist.tsv', sep='\t', index_col=0, header=None)
allc_table

In [ ]:
adata.obs['allcpath'] = f'{ENTEX_ROOT}/allc/' + adata.obs.index + '.CGN-Both.allc.tsv.gz'
# adata.obs['allcpath'] = allc_table.loc[adata.obs.index, 1]
adata.obs['coolpath'] = 'gs://ecker-zhoujt-analysis/ENTEx/tissue/' + adata.obs['Tissue'].astype(str) + '/impute/10K/' + adata.obs.index + '.cool'


In [ ]:
tmp = adata.obs[['leiden_group_mc', 'leiden_group_3c', 'allcpath', 'coolpath']].copy()
tmp['group_mc_3c'] = tmp['leiden_group_mc'].astype(str) + '-' + tmp['leiden_group_3c'].astype(str)
count = tmp['group_mc_3c'].value_counts()
tmp = tmp.loc[tmp['group_mc_3c'].isin(count.index[count>50])]
tmp[['allcpath', 'group_mc_3c']].to_csv('Mus-Skl/allclist_donor_mc3cgroup.csv', index=False, header=False)
tmp[['coolpath', 'group_mc_3c']].to_csv('Mus-Skl/coollist_donor_mc3cgroup.csv', index=True, header=False)
